In [5]:
import cv2
import mediapipe as mp
import csv
import os
import time

POSES = ["thinking", "victory", "stop"]
SAVE_DIR = "data"
os.makedirs(SAVE_DIR, exist_ok=True)

mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils

def extract_landmarks(results):
    data = []

    # Pose
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        return None  # <-- IMPORTANT FIX

    # Left hand
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * (21 * 3))

    # Right hand
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * (21 * 3))

    return data


# ---- AUTO-SAMPLING WITH CAMERA WARMUP ---- #

cap = cv2.VideoCapture(0)

print("\n⏳ Warming up camera for 10 seconds...")
warmup_start = time.time()
while time.time() - warmup_start < 10:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    cv2.putText(frame, "Warming up camera...", (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.imshow("Collecting Data", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

print("✔ Camera ready!\n")


with mp_holistic.Holistic() as holistic:
    for pose_label in POSES:

        print(f"\n🟢 Get ready for pose: {pose_label}")
        print("Holding pose in 3 seconds...")

        for i in range(3, 0, -1):
            print(i)
            time.sleep(1)

        print("Start holding the pose NOW!")
        print("Collecting for 12 seconds...")

        start_time = time.time()
        csv_path = os.path.join(SAVE_DIR, f"{pose_label}.csv")

        with open(csv_path, "w", newline="") as f:
            writer = csv.writer(f)

            while time.time() - start_time < 12:
                ret, frame = cap.read()
                if not ret:
                    break

                frame = cv2.flip(frame, 1)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(rgb)

                mp_draw.draw_landmarks(frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

                # TRY TO EXTRACT LANDMARKS
                landmarks = extract_landmarks(results)

                if landmarks is not None:
                    landmarks.append(pose_label)
                    writer.writerow(landmarks)
                    cv2.putText(frame, "Sample saved", (10, 100),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
                else:
                    cv2.putText(frame, "NO landmarks detected!", (10, 100),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

                cv2.imshow("Collecting Data", frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break

        print(f"✔ Saved samples for: {pose_label}")

cap.release()
cv2.destroyAllWindows()

print("\n🎉 Data collection completed!")



⏳ Warming up camera for 10 seconds...
✔ Camera ready!


🟢 Get ready for pose: thinking
Holding pose in 3 seconds...
3
2
1
Start holding the pose NOW!
✔ Saved samples for: thinking

🟢 Get ready for pose: victory
Holding pose in 3 seconds...
3
2
1
Start holding the pose NOW!
✔ Saved samples for: victory

🟢 Get ready for pose: stop
Holding pose in 3 seconds...
3
2
1
Start holding the pose NOW!
✔ Saved samples for: stop

🎉 Data collection completed!


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

DATA_DIR = "data"
POSES = ["thinking", "victory", "stop"]

# ---- LOAD CSVs ---- #
dfs = []
for pose in POSES:
    csv_path = os.path.join(DATA_DIR, f"{pose}.csv")
    df = pd.read_csv(csv_path, header=None)
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

print("Dataset shape:", data.shape)

# ---- SPLIT FEATURES + LABEL ---- #
X = data.iloc[:, :-1].values   # all landmark columns
y = data.iloc[:, -1].values    # the last column = pose label

# ---- TRAIN-TEST SPLIT ---- #
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

# ---- TRAIN CLASSIFIER ---- #
clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)
clf.fit(X_train, y_train)

# ---- ACCURACY ---- #
acc = clf.score(X_test, y_test)
print(f"\nTraining complete!")
print(f"Model accuracy: {acc*100:.2f}%")

# ---- SAVE MODEL ---- #
joblib.dump(clf, "pose_model.pkl")
print("\nModel saved as pose_model.pkl")


Dataset shape: (542, 226)

Training complete!
Model accuracy: 100.00%

Model saved as pose_model.pkl


In [9]:
import cv2
import mediapipe as mp
import numpy as np
import joblib

# Load model
clf = joblib.load("pose_model.pkl")

mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils

def extract_landmarks(results):
    data = []

    # Pose
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        return None

    # Left hand
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * (21 * 3))

    # Right hand
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * (21 * 3))

    return np.array(data).reshape(1, -1)


cap = cv2.VideoCapture(0)

with mp_holistic.Holistic() as holistic:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)

        mp_draw.draw_landmarks(frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
        mp_draw.draw_landmarks(frame, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_draw.draw_landmarks(frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

        # Extract landmarks
        landmarks = extract_landmarks(results)

        if landmarks is not None:
            pred = clf.predict(landmarks)[0]
            cv2.putText(frame, f"Pose: {pred}", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)
        else:
            cv2.putText(frame, "No pose detected", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,255), 3)

        cv2.imshow("Real-Time Pose Classifier", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


d:\sujal\dev\Machine learning projects\MEME CLASSIFIER\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
d:\sujal\dev\Machine learning projects\MEME CLASSIFIER\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
d:\sujal\dev\Machine learning projects\MEME CLASSIFIER\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn

In [11]:
import cv2
import mediapipe as mp
import numpy as np
import joblib
import os

# Load model
clf = joblib.load("pose_model.pkl")

# Map pose -> meme image path
MEME_MAP = {
    "thinking": "meme_images/thinking.png",
    "victory": "meme_images/victory.png",
    "stop": "meme_images/stop.png"
}

mp_holistic = mp.solutions.holistic
mp_draw = mp.solutions.drawing_utils

def extract_landmarks(results):
    data = []

    if not results.pose_landmarks:
        return None

    # Pose
    for lm in results.pose_landmarks.landmark:
        data.extend([lm.x, lm.y, lm.z])

    # Left hand
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * 63)

    # Right hand
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * 63)

    return np.array(data).reshape(1, -1)


cap = cv2.VideoCapture(0)

with mp_holistic.Holistic() as holistic:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)

        mp_draw.draw_landmarks(frame, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)

        # Extract features
        landmarks = extract_landmarks(results)

        meme_img = None
        label = "No pose"

        if landmarks is not None:
            label = clf.predict(landmarks)[0]

            # Load meme image
            meme_path = MEME_MAP.get(label, None)
            if meme_path and os.path.exists(meme_path):
                meme_img = cv2.imread(meme_path)

                # Resize meme to match webcam height
                h, w = frame.shape[:2]
                mh, mw = meme_img.shape[:2]
                scale = h / mh
                meme_img = cv2.resize(meme_img, (int(mw * scale), h))

        # If no meme image found, create empty placeholder
        if meme_img is None:
            meme_img = np.zeros_like(frame)

        # Combine (horizontal stacking)
        combined = np.hstack((frame, meme_img))

        # Add label text
        cv2.putText(combined, f"Pose: {label}",
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX,
                    1.2, (0, 255, 0), 3)

        cv2.imshow("Meme Classifier - Dual Screen", combined)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


d:\sujal\dev\Machine learning projects\MEME CLASSIFIER\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
d:\sujal\dev\Machine learning projects\MEME CLASSIFIER\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
d:\sujal\dev\Machine learning projects\MEME CLASSIFIER\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn